(Tutorial_Add_ions)=
# Add ions

*Adding monovalent ions by reproducible periodic water replacement.*

Use an already solvated system or a water box. Each added ion replaces an entire
water molecule at its oxygen position. You need one structure, one chemical
state, declared molecule membership and a finite periodic box. The input can be
any supported form supplying this information; output defaults to native MolSys.

:::{admonition} API documentation
:class: dropdown
{func}`molsysmt.build.add_ions`
:::

:::{versionadded} 1.0.0
:::

## Choosing an added population

The counts are **additional** ions, not target final populations. Existing ions
remain. The native method supports Na+, K+, Li+, Rb+, Cs+, Cl-, Br-, F- and I-.
It assigns the known ion formal charge; it does not assign partial charges or
force-field parameters, determine protonation, relax coordinates or equilibrate
salt around the solute.

In [1]:
import numpy as np
import molsysmt as msm

molsys = msm.build.make_water_box(msm.pyunitwizard.quantity(2*np.eye(3), 'nm'))
prepared = msm.build.add_ions(molsys, n_cations=2, n_anions=2,
                              random_seed=17, return_report=True)
molsys_ions = prepared['molecular_system']
print(prepared['report']['added'])
print('source/output waters:', msm.get(molsys, n_waters=True),
      msm.get(molsys_ions, n_waters=True))

{'cation': 2, 'anion': 2, 'salt_pairs': 0, 'total': 4}
source/output waters: 268 264


## Checking periodic clearances

Distances use the box row vectors and minimum periodic images, including across
opposite faces and in tilted boxes. By default each new ion is farther than
0.5 nm from every source nonwater atom and from every new/existing ion. You can
set `min_solute_distance` and `min_ion_distance` using any supported length unit.
The method does not apply a new ion-to-water packing criterion; assess the
prepared geometry and parameterize/relax it before simulation.

In [2]:
distances = msm.structure.get_distances(
    molsys_ions, selection='molecule_type == "ion"', pbc=True)
values = msm.pyunitwizard.get_value(distances, to_unit='nm')[0]
print('ion clearance respected:', bool(np.all(values[np.triu_indices(4, 1)] > 0.5)))

ion clearance respected: True


## Neutralizing a known charge

Use `'neutralize'` for both counts to add only the counterions needed by the
entire source. By default the source must have complete stored formal charges.
`charge_source='partial_charge'` instead reads a complete stored mechanical
assignment in elementary-charge units. `charge_source='physical_pH7'` explicitly
opts into a tabulated approximation; it is not an inferred chemical state.
Alternatively declare `expected_total_charge` as an integer in e or a charge
quantity. A declaration conflicting with complete formal charges fails.
Fractional neutralizing totals outside 1e-4 e of an integer fail.

In a mixed request, for example two explicit cations and automatic anions, the
automatic species must cancel the source charge **plus** the explicit ions.
An impossible sign combination fails. Declared waters must be neutral: known
charged waters or covalent links to another molecule are rejected. Missing
water charges are not filled in or advertised as a measured assignment.

In [3]:
neutral = msm.build.add_ions(molsys, n_cations='neutralize', n_anions='neutralize',
                             expected_total_charge=0, random_seed=17, return_report=True)
print(neutral['report']['charge'])

{'source': 'caller_declaration', 'total_elementary_charge': 0, 'output_total_elementary_charge': 0}


## Adding salt concentration

`ionic_strength` adds equal positive/negative pairs **after** the base counts.
The water-count convention is round-half-up of
`remaining_waters * concentration_in_molar / 55.4`.
Base replacements are excluded from the remaining-water count; existing and
neutralizing ions are excluded from the requested salt contribution. This is a
concentration proxy inspired by OpenMM, not a measured final ionic strength or
an exact box-volume concentration.

In [4]:
salt = msm.build.add_ions(molsys, ionic_strength='150 millimolar',
                          random_seed=17, return_report=True)
print(salt['report']['added'])

{'cation': 1, 'anion': 1, 'salt_pairs': 1, 'total': 2}


## Keeping correspondence and evidence

The report uses `molsysmt.ion_replacement@1`. A removed source atom maps to -1;
an added output atom has no original source index and maps to -1 in reverse.
Compose it with other preparation/export maps using
{func}`molsysmt.basic.compose_atom_maps`. Map entries are **indices**, not IDs.
The preparation history in native ChemicalStates retains a detached report;
H5MSM 0.5 preserves it without needing MolecularMechanics persistence.
Later extraction keeps its original **operation-scoped** axes, not a new map
for the extracted system. Other output forms can discard this native evidence.

When atoms change, named analyses are remapped and their evaluated coverage is
invalidated conservatively. Recalculate them before reuse. A zero-addition copy
preserves coverage. Atom-growth policies clear unsupported per-atom mechanical
or structural observations with diagnostics; retained coordinates and box values
are unchanged.

In [5]:
report = prepared['report']
atom_map = msm.basic.compose_atom_maps([report])
print('removed source atoms:', int(np.sum(atom_map['source_to_output_atom_indices'] < 0)))
print('new output atoms:', int(np.sum(atom_map['output_to_source_atom_indices'] < 0)))
print(report['random'])

removed source atoms: 12
new output atoms: 4
{'generator': 'PCG64', 'seed': '17', 'attempts': 1, 'max_attempts': 10}


## Reproducing placement and understanding failure

The PCG64 seed, software versions and distances in nm are retained. With no
seed a fresh seed is generated and recorded as a decimal string. Reuse that
integer with the same ordered input, parameters and software to reproduce a run.
`max_attempts` bounds independent random greedy attempts. Failure leaves the
source unchanged and can occur even if another packing exists. Lower the counts
or clearances, enlarge the cell, or choose another seed deliberately.

Methodological inspiration is pinned to
[OpenMM Modeller._addIons](https://github.com/openmm/openmm/blob/ba0aa236ac4d07241382869604c4d2ea58c4f3f5/wrappers/python/openmm/app/modeller.py#L259).
This native implementation adds PBC-aware shared cell lists, explicit seeds and
bounded attempts; it does not claim parity with all OpenMM criteria. Bibliography
is retained independently of optional Ackredit tracking. Runtime credit separates
executed software from methodological inspiration.

In `solvate`, OpenMM explicit counts or a seed use this native placement after
water-only construction. Unseeded default neutralization/concentration keeps the
OpenMM provider route. Native solvation delegates placement here while retaining
its documented legacy pH-7 charge approximation. PDBFixer currently rejects
explicit counts or a seed before construction; use OpenMM/native solvation or
call this operation on an independently prepared water system.

:::{seealso}
:class: dropdown
- {ref}`Solvate <Tutorial_Solvate>`
- {ref}`Make water box <Tutorial_Make_water_box>`
- {func}`molsysmt.basic.compose_atom_maps`
:::